In [2]:
import numpy as np 

In [3]:
class Layer:
    def __init__(self, row, cols):
        self.weights = 0.01 * np.random.randn(row, cols)
        self.bias = np.zeros((1, cols))

    def forward(self, inputs):
        self.x = inputs
        self.output = np.dot(self.x, self.weights) + self.bias
        return self.output

    def backward(self, dvalues):
        self.dL_dz = dvalues
        self.dweights = np.dot(self.x.T, self.dL_dz  )
        self.dbias = np.sum(self.dL_dz, axis = 0, keepdims= True)
        self.dinputs = np.dot(self.dL_dz, self.weights.T)

In [4]:
class relu:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        return self.output

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

In [6]:
class CrossEntropyLoss:
    def forward(self, y_pred, y_true):
        y_pred_cliped = np.clip(y_pred, 1e-5, 1-1e-5 )
        # check for one hot encoding 
        if (len(y_true.shape)== 1):
            # Not One hot encoded 
            y_true = np.eye(len(y_pred[0]))[y_true]

        
        correct_confidences = np.sum(y_true * np.log(y_pred_cliped), axis=1, keepdims= True)

        neg_loss = - correct_confidences
        avg_loss = np.mean(neg_loss)
        return avg_loss

    def backward(self, dvalues , y_true):
        if (len(y_true.shape)== 1):
            y_true = np.eye(len(dvalues[0]))[y_true]

        dvalues = np.clip(dvalues, 1e-5, 1-1e-5)
        
        self.dinputs = - y_true/dvalues
        self.dinputs = self.dinputs/(len(dvalues))

In [7]:
# Softmax activation
class Activation_Softmax:
 # Forward pass
 def forward(self, inputs):
 # Get unnormalized probabilities
  exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
 # Normalize them for each sample
  probabilities = exp_values / np.sum(exp_values, axis=1,keepdims=True)
  self.output = probabilities

In [8]:
class Activation_Softmax_crossEntropy_loss:
    def __init__(self):
        self.softmax = Activation_Softmax()
        self.cross = CrossEntropyLoss()

    def forward(self, inputs, y_true):
        self.softmax.forward(inputs)
        self.outputs = self.cross.forward(self.softmax.output, y_true)
        return self.outputs

    def backward(self, predicted , y_true):
        samples = len(predicted)
        # if the y_true is one hot encoded
        if(len(y_true.shape) == 2):
            y_true = np.argmax(y_true, axis = 1)
        
        self.dinputs = predicted.copy()
        self.dinputs[range(len(predicted)), y_true] -= 1
        self.dinputs = self.dinputs/samples

In [10]:
class optimizer_adam:
    def __init__(self, learning_rate = 1.0, decay = 0., momentum = 0.9 ,epsilon = 1e-5,  beta1 = 0.9 , beta2= 0.999 ):
        self.learning_rate = learning_rate
        self.current_learning_rate = self.learning_rate
        self.decay = decay 
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon 
        self.iterations = 0

    def pre_updates(self):
        if self.decay: 
            self.current_learning_rate = self.learning_rate/ (1 + self.decay * self.iterations)

    def update_params(self, layer):
        if not hasattr(layer, "weight_momentums"):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.bias)
            layer.bias_cache = np.zeros_like(layer.bias)


        layer.weight_cache = self.beta2 * layer.weight_cache + (1 - self.beta2)* layer.dweights**2
        layer.bias_cache = self.beta2 * layer.bias_cache + (1 - self.beta2)* layer.dbias**2

        layer.weight_momentums = self.beta1 * layer.weight_momentums + (1 - self.beta1)* layer.dweights
        layer.bias_momentums = self.beta1 * layer.bias_momentums + (1 -self.beta1) * layer.dbias

        weights_momentum_corrected = layer.weight_momentums /(1 - self.beta1**(self.iterations + 1))
        bias_momentum_corrected = layer.bias_momentums/ (1 - self.beta1**(self.iterations + 1))

        weights_cache_corrected = layer.weight_cache/ (1 - self.beta2**(self.iterations + 1))
        bias_cache_corrected = layer.bias_cache/ (1 - self.beta2** (self.iterations + 1))

        layer.weights += -self.current_learning_rate * weights_momentum_corrected/(np.sqrt(weights_cache_corrected) + self.epsilon)
        layer.bias += - self.current_learning_rate * bias_momentum_corrected/(np.sqrt(bias_cache_corrected) + self.epsilon)

    def post_updates(self):
        self.iterations +=1


In [12]:
from nnfs.datasets import spiral_data
X, y = spiral_data(samples = 100 , classes =  3)


In [18]:
dense1 = Layer(2, 64)
activation1 = relu()
dense2 = Layer(64, 3)
loss_func = Activation_Softmax_crossEntropy_loss()
optimizer = optimizer_adam(learning_rate=0.02, decay=5e-7)


for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    loss = loss_func.forward(dense2.output, y)

    predictions = np.argmax(loss_func.softmax.output, axis = 1 )
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    
    accuracy = np.mean(predictions == y)

    if not epoch% 100 :
        print(f"iteration = {epoch}, accuracy = {accuracy}, loss = {loss}, lr = {optimizer.current_learning_rate}")


    loss_func.backward( loss_func.softmax.output, y)
    dense2.backward(loss_func.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_updates()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_updates()

iteration = 0, accuracy = 0.3566666666666667, loss = 1.0986265986739636, lr = 0.02
iteration = 100, accuracy = 0.5066666666666667, loss = 0.9154303602839927, lr = 0.019999010049002574
iteration = 200, accuracy = 0.74, loss = 0.6651333640067939, lr = 0.019998010197985302
iteration = 300, accuracy = 0.8, loss = 0.5320778846434336, lr = 0.019997010446938183
iteration = 400, accuracy = 0.8266666666666667, loss = 0.46229024388036055, lr = 0.01999601079584623
iteration = 500, accuracy = 0.8433333333333334, loss = 0.4182524611920063, lr = 0.01999501124469445
iteration = 600, accuracy = 0.8566666666666667, loss = 0.3877214131238402, lr = 0.01999401179346786
iteration = 700, accuracy = 0.8666666666666667, loss = 0.3647368457655215, lr = 0.01999301244215147
iteration = 800, accuracy = 0.8666666666666667, loss = 0.34791843681583223, lr = 0.019992013190730303
iteration = 900, accuracy = 0.89, loss = 0.3336200271896964, lr = 0.019991014039189386
iteration = 1000, accuracy = 0.8866666666666667, loss